# Demo del Sistema Inteligente de Alerta Temprana

## Objetivo

Este notebook presenta una demostración interactiva del sistema desarrollado para el proyecto **Sistema Inteligente de Alerta Temprana para Crisis Educativa Municipal**.

A partir de un municipio seleccionado por el usuario, el sistema:

- Consulta los indicadores educativos del municipio.
- Estima su tasa de deserción mediante el modelo Random Forest entrenado.
- Clasifica automáticamente el nivel de riesgo.
- Simula una intervención educativa.
- Estima el impacto potencial de dicha intervención sobre la deserción escolar.

## Carga de librerías

In [2]:
#Librerias
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import joblib
import numpy as np
import pandas as pd

import ipywidgets as widgets
from IPython.display import display

## Carga del proyecto

Se localiza automáticamente la carpeta principal del proyecto para cargar el conjunto de datos y el modelo entrenado.

In [3]:
PROJECT_ROOT = Path.cwd()

while PROJECT_ROOT.name != "educational_crisis_ai":
    PROJECT_ROOT = PROJECT_ROOT.parent

print(PROJECT_ROOT)

/home/juan/Downloads/educational_crisis_ai/educational_crisis_ai


## Carga del conjunto de datos y del modelo

Se carga el conjunto de datos procesado utilizado por el modelo y el modelo Random Forest previamente entrenado.

In [4]:
df = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "dataset_modelado.csv"
)

modelo = joblib.load(
    PROJECT_ROOT / "models" / "random_forest_desercion.pkl"
)

print("Dataset:", df.shape)
print("Modelo cargado correctamente")

Dataset: (15707, 85)
Modelo cargado correctamente


## Ejecución de la simulación

In [81]:
def ejecutar_simulacion(b):

    with salida:

        clear_output()

        municipio_demo = selector.value

        escenario = df[
            (df["AÑO"] == 2024) &
            (df["MUNICIPIO"].str.strip().str.upper() == municipio_demo.strip().upper())
        ].copy()

        if escenario.empty:
            print("❌ Municipio no encontrado.")
            return

        escenario = escenario.iloc[[0]]

        print("=" * 60)
        print("MUNICIPIO SELECCIONADO")
        print("=" * 60)

        print(f"Municipio   : {escenario['MUNICIPIO'].values[0]}")
        print(f"Departamento: {escenario['DEPARTAMENTO'].values[0]}")
        print(f"Año         : {escenario['AÑO'].values[0]}")

        print("\nINDICADORES EDUCATIVOS")
        print("-" * 60)

        print(f"Cobertura neta : {escenario['COBERTURA_NETA'].values[0]:.2f}%")
        print(f"Aprobación     : {escenario['APROBACIÓN'].values[0]:.2f}%")
        print(f"Reprobación    : {escenario['REPROBACIÓN'].values[0]:.2f}%")
        print(f"Repitencia     : {escenario['REPITENCIA'].values[0]:.2f}%")
        print(f"Deserción real : {escenario['DESERCIÓN'].values[0]:.2f}%")

        # ----------------------------
        # Preparar datos para el modelo
        # ----------------------------

        columnas_excluir = [
            "DESERCIÓN",
            "MUNICIPIO",
            "DEPARTAMENTO",
            "CÓDIGO_MUNICIPIO",
            "DESERCIÓN_ETC",
            "DESERCIÓN_TRANSICIÓN",
            "DESERCIÓN_TRANSICIÓN_ETC",
            "DESERCIÓN_PRIMARIA",
            "DESERCIÓN_PRIMARIA_ETC",
            "DESERCIÓN_SECUNDARIA",
            "DESERCIÓN_SECUNDARIA_ETC",
            "DESERCIÓN_MEDIA",
            "DESERCIÓN_MEDIA_ETC",
            "ETC_ETC"
        ]

        X_actual = escenario.drop(columns=columnas_excluir, errors="ignore")
        X_actual = X_actual[modelo.feature_names_in_]

        pred_actual = modelo.predict(X_actual)[0]

        # ----------------------------
        # Simulación de intervención
            # ----------------------------

        escenario_intervenido = escenario.copy()

        escenario_intervenido["COBERTURA_NETA"] += cobertura.value
        escenario_intervenido["APROBACIÓN"] += aprobacion.value
        escenario_intervenido["REPROBACIÓN"] += reprobacion.value
        escenario_intervenido["REPITENCIA"] += repitencia.value

        X_intervenido = escenario_intervenido.drop(columns=columnas_excluir, errors="ignore")
        X_intervenido = X_intervenido[modelo.feature_names_in_]

        pred_intervenido = modelo.predict(X_intervenido)[0]

        def riesgo(valor):
            if valor < 4:
                return "🟢 Bajo"
            elif valor < 8:
                return "🟡 Medio"
            else:
                return "🔴 Alto"

        print("\n" + "=" * 60)
        print("PREDICCIÓN DEL MODELO - TASA DE DESERCIÓN (%)")
        print("=" * 60)

        print(f"Escenario actual      : {pred_actual:.2f}% ({riesgo(pred_actual)})")
        print(f"Con intervención      : {pred_intervenido:.2f}% ({riesgo(pred_intervenido)})")
        print(f"Reducción estimada    : {pred_actual - pred_intervenido:.2f} puntos porcentuales")

        print("\nINTERVENCIÓN APLICADA")
        print("-" * 60)
        print(f"Cobertura neta : {cobertura.value:.1f}%")
        print(f"Aprobación     : {aprobacion.value:.1f}%")
        print(f"Reprobación    : {reprobacion.value:.1f}%")
        print(f"Repitencia     : {repitencia.value:.1f}%")

## Configuración de la intervención

Los controles deslizantes permiten modificar los principales indicadores educativos del municipio seleccionado. Estos cambios representan un escenario hipotético de intervención cuyos efectos serán evaluados por el modelo.

In [82]:
cobertura = widgets.FloatSlider(
    value=5,
    min=-10,
    max=10,
    step=0.5,
    description="Cobertura"
)

aprobacion = widgets.FloatSlider(
    value=3,
    min=-10,
    max=10,
    step=0.5,
    description="Aprobación"
)

reprobacion = widgets.FloatSlider(
    value=-2,
    min=-10,
    max=10,
    step=0.5,
    description="Reprobación"
)

repitencia = widgets.FloatSlider(
    value=-1,
    min=-10,
    max=10,
    step=0.5,
    description="Repitencia"
)

display(cobertura)
display(aprobacion)
display(reprobacion)
display(repitencia)

FloatSlider(value=5.0, description='Cobertura', max=10.0, min=-10.0, step=0.5)

FloatSlider(value=3.0, description='Aprobación', max=10.0, min=-10.0, step=0.5)

FloatSlider(value=-2.0, description='Reprobación', max=10.0, min=-10.0, step=0.5)

FloatSlider(value=-1.0, description='Repitencia', max=10.0, min=-10.0, step=0.5)

In [83]:
boton.on_click(ejecutar_simulacion)

## Resultados de la simulación

El sistema presenta los indicadores actuales del municipio, estima la tasa de deserción para el escenario actual y compara dicho resultado con el escenario intervenido definido por el usuario.

In [ ]:
# Municipios disponibles (2024)
municipios = sorted(
    df.loc[df["AÑO"] == 2024, "MUNICIPIO"]
      .dropna()
      .unique()
)

selector = widgets.Dropdown(
    options=municipios,
    description="Municipio:",
    layout=widgets.Layout(width="450px")
)

boton = widgets.Button(
    description="Ejecutar simulación",
    button_style="success"
)

salida = widgets.Output()

display(selector)
display(boton)
display(salida)

Dropdown(description='Municipio:', layout=Layout(width='450px'), options=('Abejorral', 'Abrego', 'Abriaquí', '…

Button(button_style='success', description='Ejecutar simulación', style=ButtonStyle())

Output()